# `GenMeshMaker` — surface plotting cookbook

`GenMeshMaker` (in [`dpu_mini/mesh_maker.py`](../dpu_mini/mesh_maker.py)) wraps a FreeSurfer subject's
meshes (`pial`, `inflated`, `sphere`) and surface properties (`curv`, `thickness`) so you can:

- map per-vertex scalar data, RGB(A) data, or ROI labels onto a colour-mapped surface (`return_display_rgb`)
- flatten a patch of cortex (e.g. visual cortex) into 2D and plot it with matplotlib (`make_flat_map` + `flat_mpl`)
- plot ROI boundaries on top of any of the above
- open in freesurfer
- (via the `MeshDash` subclass) render an interactive 3D surface with Plotly

This notebook walks through each of those capabilities on a real subject, using the Benson14
visual-field atlas (eccentricity / polar angle / visual area labels) as realistic example data,
since it's already retinotopically informative and ships with FreeSurfer's `recon-all`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from dpu_mini.mesh_maker import GenMeshMaker
from dpu_mini.mesh_format import load_benson14_info
from dpu_mini.fs_tools import dpu_load_roi, dpu_roi_list_expand

## 1. Instantiate

`GenMeshMaker(sub, fs_dir)` immediately loads, for **both hemispheres**:
- `mesh_info['inflated' | 'sphere' | 'pial']` — each a dict of `coords`/`faces` plus flattened
  `x,y,z,i,j,k` (handy for plotly `Mesh3d`). By default the hemispheres are offset along x so a
  combined lh+rh mesh doesn't overlap (`do_offsets=True`).
- `us_values['curv' | 'thickness']` and pre-computed greyscale `us_cols[...]` — the "undersurface"
  background that data gets blended over.

In [ ]:
sub    = 'sub-C001'
fs_dir = os.environ['SUBJECTS_DIR']   # FreeSurfer SUBJECTS_DIR

gm = GenMeshMaker(sub=sub, fs_dir=fs_dir)

print('vertices per hemi:', gm.n_vx, ' total:', gm.total_n_vx)
print('meshes loaded:', list(gm.mesh_info.keys()))
print('inflated/rh keys:', list(gm.mesh_info['inflated']['rh'].keys()))
print('undersurfaces cached:', list(gm.us_values.keys()))

In [ ]:
# Real example data: Benson14 visual-field atlas (already run for this subject)
#   ecc   - eccentricity, degrees of visual angle
#   pol   - polar angle, degrees
#   varea - visual area label (0 = not visual cortex, 1=V1, 2=V2, 3=V3, ...)
b14   = load_benson14_info(sub, fs_dir)
ecc, pol, varea = b14['ecc'], b14['pol'], b14['varea']
print(f'{(varea > 0).sum()} / {len(varea)} vertices fall inside the visual-field atlas')

## 2. Flatten a patch of cortex

`flat_mpl` (matplotlib 2D plotting) needs a flattened mesh to draw triangles on. There's no
`lh.flat`/`rh.flat` from `mris_flatten` for this subject, so `make_flat_map` builds a quick
spherical-projection flattening ourselves — not as good as a proper FreeSurfer flattening, but
enough to look at a patch of cortex.

We centre it on the visual cortex (`varea != 0`) and cut it down to a circular patch
(`cut_circle=0.6`) so it isn't swamped by unrelated cortex. The result is stored as
`gm.mesh_info['flat']` and reused by every `flat_mpl` call below.

In [ ]:
gm.make_flat_map(
    varea != 0,
    hemi_project='sphere',   # project from the spherical mesh
    cut_circle=0.6,          # keep a circular patch around the centre
)
print(gm.mesh_info['flat']['rh']['coords'].shape, gm.mesh_info['flat']['rh']['faces'].shape)

## 3. Just the undersurface

Calling `flat_mpl()` with no `data` falls back to plotting the undersurface only
(`under_surf='curv'` by default — a binarised sulc/gyrus mask). This is the "blank canvas" every
other plot gets blended over.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
gm.flat_mpl(under_surf='curv', hemi_list=['rh'], ax=axs[0], colorbar=False)
axs[0].set_title('under_surf = curv')
gm.flat_mpl(under_surf='thickness', hemi_list=['rh'], ax=axs[1], colorbar=False)
axs[1].set_title('under_surf = thickness')
plt.show()

## 4. Scalar data + colormap

Pass a 1D array (one value per vertex, length `total_n_vx`) plus `cmap`/`vmin`/`vmax`. Here we
plot real cortical thickness (already cached in `gm.us_values['thickness']`, but you'd typically
pass your own per-vertex stat map).

In [ ]:
thickness = gm.get_us_values('thickness')   # (total_n_vx,) mm

gm.flat_mpl(
    data=thickness,
    cmap='viridis', vmin=0, vmax=4,
    hemi_list=['rh'],
    surf_name='cortical thickness (mm)',
)
plt.show()

## 5. Masking with `data_mask`

`data_mask` hides vertices outside a region — masked vertices fall back to the undersurface
instead of being coloured. Below we restrict the same thickness map to area V1
(loaded as a FreeSurfer label via `dpu_load_roi`).

Note the trailing `.` on `'b14_V1.'` — `b14_V1V2V3.label` also exists for this subject and would
otherwise substring-match `'b14_V1'` too; `dpu_roi_list_expand` (section 6) handles this
disambiguation for you automatically.

In [ ]:
v1_bool = dpu_load_roi(sub, 'b14_V1.', fs_dir=fs_dir)   # boolean, length total_n_vx
print(f'V1: {v1_bool.sum()} vertices')

gm.flat_mpl(
    data=thickness, data_mask=v1_bool,
    cmap='viridis', vmin=0, vmax=4,
    hemi_list=['rh'],
    surf_name='thickness, V1 only',
)
plt.show()

## 6. ROI borders with `roi_list`

`roi_list` draws ROI outlines on top of whatever's plotted underneath, instead of filling them in.
`dpu_roi_list_expand` finds every `b14_*` label for this subject and disambiguates overlapping
names, so we get one outline per visual area.

In [ ]:
roi_list = dpu_roi_list_expand(sub, 'b14_', fs_dir)
print(roi_list)

gm.flat_mpl(
    data=thickness,
    cmap='viridis', vmin=0, vmax=4,
    hemi_list=['rh'],
    roi_list=roi_list,
    surf_name='thickness, all Benson14 ROI borders',
)
plt.legend(bbox_to_anchor=(1.3, 1), fontsize=7)
plt.show()

## 7. `data_sub_mask` — data that only exists for a subset of vertices

Sometimes you only *have* values for a subset of vertices (e.g. a pRF fit restricted to V1) rather
than a full-length array with most of it masked out. `data_sub_mask` takes a short array (one
value per `True` in the mask) and scatters it back into a full-length array for you.

Note that you can use any matplotlib default, or make your own colour map  — `_data_to_rgb` falls back to
`dpu_cmap_from_str` whenever `dpu_get_cmap` doesn't recognise the name.

In [ ]:
ecc_in_v1 = ecc[v1_bool]   # short array: one eccentricity value per V1 vertex
print(ecc_in_v1.shape, '->', v1_bool.sum(), 'True entries in the mask')

gm.flat_mpl(
    data=ecc_in_v1, data_sub_mask=v1_bool,
    cmap='jet', vmin=0, vmax=20,
    hemi_list=['rh'],
    roi_list=['b14_V1.'],
    surf_name='eccentricity (deg), V1 only',
)
plt.show()

## 8. Direct RGB(A) — bypass the colormap entirely

If `data` already has shape `(n_vx, 3)` or `(n_vx, 4)`, `return_display_rgb`/`flat_mpl` use it as
literal colour instead of running it through a colormap. This is how `surface_image.ipynb` paints
a photo onto the cortex pixel-by-pixel; here's the same mechanism with a simple synthetic
checkerboard so it's easy to see exactly what's happening.

In [ ]:
y_all = np.concatenate([gm.mesh_info['inflated']['lh']['y'], gm.mesh_info['inflated']['rh']['y']])
z_all = np.concatenate([gm.mesh_info['inflated']['lh']['z'], gm.mesh_info['inflated']['rh']['z']])
checker = ((y_all // 10).astype(int) + (z_all // 10).astype(int)) % 2

rgb = np.stack([checker, 1 - checker, np.full_like(checker, 0.3)], axis=1)  # (total_n_vx, 3)

gm.flat_mpl(
    data=rgb, 
    data_mask=v1_bool, 
    colorbar=False, 
    surf_name='checkerboard (direct RGB)')
plt.show()

## 9. Pulling out a sub-mesh

`return_sm(mask, hemi)` extracts just the vertices/faces inside a mask as their own small mesh
(with re-indexed faces) — useful for, say, exporting one ROI in isolation or running geometry
ops on a region without the whole hemisphere. `return_pyc_sm` wraps the same thing as a
[pycortex](https://github.com/gallantlab/pycortex) `Surface` (needed for things like geodesic
distance).

In [ ]:
v1_submesh = gm.return_sm(v1_bool, hemi='rh', surf='inflated')
print('V1 sub-mesh:', v1_submesh['coords'].shape[0], 'vertices,', v1_submesh['faces'].shape[0], 'faces')
print('(full rh inflated mesh has', gm.n_vx['rh'], 'vertices for comparison)')

# 10. Interacting with Freesurfer

* requires freeview 
* scripts will create a custom surf file, and the command (which contains the colormap info) to open it in freeview
* The colormap can be anything from matplotlib. Just specify the min and max values. (https://matplotlib.org/stable/tutorials/colors/colormaps.html)
* You can also specify the camera angle for when freeview opens, and ask it to automatically take a picture of the surface. This can be useful if you want to iterate through several subjects/surface plots and save the figures as pngs, but can't be bothered to sit and click again and again... 

In [ ]:
# Add polar angle plot
gm.add_surface(
    data = pol,
    surf_name = f'{sub}-polar_angle',    
    data_mask=v1_bool,
    cmap='twilight', vmin=0, vmax=180,
    
)

# Add eccentricity
gm.add_surface(
    data = ecc,
    surf_name = f'{sub}-eccentricity',    
    cmap='jet', vmin=0, vmax=20,
    data_mask=v1_bool,
)

In [ ]:
# Now we can open one of the surfaces in freeview
gm.open_fs_surface(
    surf_name=f'{sub}-polar_angle',
    mesh = 'inflated',          # what type of surface? inflated? pial?
    )

I've written many possible things you can do with this. See the notes for 'write_fs_cmd'

For write_fs_cmd
        '''
        Write the bash command to open the specific surface with the overlay

        **kwargs 
        surf_name       which surface(s) to open (of the custom ones you have made)
        mesh_list       which mesh(es) to plot the surface info on (e.g., inflated, pial...)
        hemi_list       which hemispheres to load
        roi_list        which roi outlines to load
        roi_col_spec    if loading rois, what color? If not specified will do different colors for each nes     
        roi_mask        mask by roi?
        keep_running    keep running the command (use "&" at the end of the command). Useful if you want to take many screen shots.
        shading_off     Turn of shading? i.e., don't make it darker underneath. Default is false        
        do_scrn_shot    bool            take a screenshot of the surface when it is loaded?
        scr_shot_file   str             Where to put the screenshot. If not specified goes in custom surface dir
        azimuth         float           camera angle(0-360) Default: 0
        zoom            float           camera zoom         Default: 1.00
        elevation       float           camera angle(0-360) Default: 0
        roll            float           camera angle(0-360) Default: 0        
        do_col_bar      bool            show color bar at the end. Default is true
        '''

See also dpu_make_overlay_str

In [ ]:
# Maybe we want to open with a specific cameram angle and take a screenshot?
scr_shot_file = './z_screenshots'
if not os.path.exists(scr_shot_file):
    os.makedirs(scr_shot_file)
scr_shot_file = os.path.abspath(scr_shot_file)
gm.open_fs_surface(
    surf_name=f'{sub}-polar_angle',
    mesh = 'inflated',          # what type of surface? inflated? pial?
    hemi_list= 'lh', 
    do_scr_shot = True,
    scr_shot_file = os.path.join(scr_shot_file,'eg.png'), # Where to put it?
    # *** camera angles ***
    azimuth = 140, zoom = 1, elevation=5, roll=0, 
    )

## 10. Exporting a coloured surface to `.ply`

`add_ply_surface` writes `lh.<name>.ply` / `rh.<name>.ply` into `output_dir`, with the same
`return_display_rgb` colouring kwargs as everything above baked into per-vertex colour. These
open directly in MeshLab, Blender, or any standard 3D viewer — handy for sharing a result with
someone who doesn't have this codebase.

In [ ]:
gm.output_dir = '/tmp/genmeshmaker_demo_out'
os.makedirs(gm.output_dir, exist_ok=True)
gm.ply_files = {}

gm.add_ply_surface(
    data=thickness, surf_name='thickness_demo',
    mesh_name='inflated', cmap='viridis', vmin=0, vmax=4,
)
print(os.path.join(gm.output_dir, 'lh.thickness_demo.ply'))

## 11. Bonus: interactive 3D with `MeshDash`

**NOTE THIS CAN GET HEAVY WITH CACHE - remember to delete and be cautious with git**


`MeshDash` (in [`dpu_mini/mesh_dash.py`](../dpu_mini/mesh_dash.py)) subclasses `GenMeshMaker` and
adds `add_plotly_surface`, which builds the same `return_display_rgb` colouring but on a real,
rotatable 3D `Mesh3d` instead of a flattened 2D patch. Same kwargs throughout (`data`, `data_mask`,
`cmap`, `vmin`, `vmax`, ...).

Passing `return_mesh_obj=True` returns the raw `Mesh3d` trace(s) instead of a full `Figure`, so we
can drop eccentricity and polar angle into side-by-side 3D scenes with `plotly.subplots`. 

In [ ]:
import gc, matplotlib.pyplot as plt
try:
    import psutil; rss = lambda: psutil.Process().memory_info().rss/1e6
except ImportError:
    rss = lambda: float('nan')
gc.collect()
print(f"RSS={rss():.0f} MB | open mpl figs={len(plt.get_fignums())} | Out cached={len(Out)}")

from dpu_mini.mesh_dash import MeshDash
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, Patch

md_ = MeshDash(sub=sub, fs_dir=fs_dir)

ecc_mesh = md_.add_plotly_surface(
    data=ecc, 
    data_mask=varea > 0, 
    mesh_name='inflated', cmap='ecc', vmin=0, vmax=10,
    hemi_list=['rh'], return_mesh_obj=True,
)
pol_mesh = md_.add_plotly_surface(
    data=pol, data_mask=varea > 0, 
    mesh_name='inflated', cmap='twilight', vmin=0, vmax=180,
    hemi_list=['rh'], return_mesh_obj=True,
)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('eccentricity (deg)', 'polar angle (deg)'),
)
for trace in ecc_mesh:
    fig.add_trace(trace, row=1, col=1)
for trace in pol_mesh:
    fig.add_trace(trace, row=1, col=2)

scene_axis = dict(showgrid=False, showticklabels=False, title='', showbackground=False)
fig.update_layout(
    height=500, width=1000, showlegend=False,
    scene=dict(xaxis=scene_axis, yaxis=scene_axis, zaxis=scene_axis, uirevision='constant'),
    scene2=dict(xaxis=scene_axis, yaxis=scene_axis, zaxis=scene_axis, uirevision='constant'),
)

fig.show('browser')
# fig.write_html('example_html.html') # To save as html

More interactive plotter - add multiple maps; flip between etc. 

In [ ]:

fs = MeshDash(
    sub, 
    fs_dir=fs_dir,
    output_dir='z_dash',
    )
fs.web_get_ready()
fs.web_add_vx_col(
    data=pol,
    data4mask = v1_bool,
    cmap='twilight', vmin=0, vmax=180,
    vx_col_name=f'pol'
)

fs.web_add_vx_col(
    data=ecc, 
    data4mask =v1_bool,
    cmap='jet', 
    vmin=0, vmax=40, 
    vx_col_name=f'ecc'
)

# Other cool things....
# you can add ROIs
fs.web_add_roi(roi_list='V1')

pol_rad = np.deg2rad(pol)

hemi_sign = np.ones(len(ecc))
hemi_sign[gm.n_vx['lh']:] = -1          # rh sees left visual field

x_vis = ecc * np.sin(pol_rad) * hemi_sign * -1
y_vis = ecc * np.cos(pol_rad)

# You can also add conditional functions to do plotting. 
# For example if you have a function to plot the timeseries put that here 
def my_plot_function(vertex_id):
    fig, ax = plt.subplots(1,1,figsize=(5,5))
    ax.scatter(x_vis[vertex_id], y_vis[vertex_id])
    ax.set_ylim(-80,80)
    ax.set_xlim(-80,80)

    # Do some plotting based on the vertex...
    return fig

fs.web_add_mpl_fig_maker(my_plot_function) # you can add more than one...



In [ ]:
# To launch app from here 
app = fs.web_launch_with_dash() 
# Open the app in a browser (or in the notebook)
app.run(mode='external', host='127.0.0.1', port=8000, debug=False, use_reloader=False) 
# Go to localhost:8000 in your browser to see the app.

## Method map

| Want to... | Use |
|---|---|
| Plot scalar data on a flattened patch | `make_flat_map(...)` once, then `flat_mpl(data=..., cmap=..., vmin=..., vmax=...)` |
| Hide vertices outside a region | `data_mask=` |
| Plot data that's only defined for a subset of vertices | `data_sub_mask=` |
| Paint literal colours instead of a colormap | pass `data` with shape `(n_vx, 3)` or `(n_vx, 4)` |
| Outline ROI borders | `roi_list=[...]` (fill instead with `roi_fill=True`) |
| Get raw per-vertex RGBA without plotting | `return_display_rgb(...)` |
| Extract a region as its own mesh | `return_sm(mask, hemi)` / `return_pyc_sm(...)` |
| Export a coloured surface for MeshLab/Blender | `add_ply_surface(data=..., surf_name=...)` |
| Interactive rotatable 3D | `MeshDash(...).add_plotly_surface(...)` |
